# FM Receiver from IQ Samples

This notebook builds a simple software receiver around either a local IQ capture or a synthetic fallback signal. It is the first practical notebook that runs end-to-end from RF-ish samples to recovered audio.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from rf_utils import *
from IPython.display import Audio, Markdown, display
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from scipy import signal

%matplotlib widget

plt.rcParams.update({
    "figure.figsize": (12, 4),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})


In [ ]:
CAPTURE_ROOT = ROOT / "assets" / "local"
capture_status = probe_rtlsdr()
display(Markdown(
    f"**RTL-SDR status:** installed={capture_status['installed']}, "
    f"available={capture_status['available']}. {capture_status['message']}"
))


## Capture Source

If `assets/local/fm_receiver_iq.npz` exists, it will be used. Otherwise this notebook synthesizes a narrowband FM signal at an offset inside a complex baseband recording.

In [ ]:
capture_path = CAPTURE_ROOT / "fm_receiver_iq.npz"
if capture_path.exists():
    fs_iq, iq = load_complex_capture(capture_path)
    print(f"Loaded local capture: {capture_path.name}, fs={fs_iq}")
else:
    fs_iq = 240_000
    t = np.arange(0, 3.0, 1 / fs_iq)
    message = normalize(
        0.7 * np.sin(2 * np.pi * 700 * t)
        + 0.3 * np.sin(2 * np.pi * 1400 * t)
        + 0.2 * np.sin(2 * np.pi * 2200 * t)
    )
    iq = synthesize_fm_iq(message, fs=fs_iq, carrier_offset=45_000, freq_dev=2500)
    iq += 0.15 * synthesize_fm_iq(0.5 * np.sin(2 * np.pi * 1200 * t), fs=fs_iq, carrier_offset=-30_000, freq_dev=1800)
    iq = normalize(iq)
    print("Using synthetic FM IQ fallback.")


In [ ]:
audio_out = audio_output_widget()
fig, axes = plt.subplots(2, 2, figsize=(13, 7))

def update_receiver(tune_offset=45_000.0, channel_bw=12_500.0):
    shifted = complex_mix_down(iq, fs=fs_iq, freq_shift=tune_offset)
    filtered = lowpass_filter(shifted, cutoff_hz=channel_bw / 2, fs=fs_iq, order=5)
    audio = fm_demodulate_iq(filtered, fs=fs_iq, audio_cutoff=4000)

    for ax in axes.flat:
        ax.clear()
    plot_spectrum(iq.real, fs=fs_iq, ax=axes[0, 0], title="I-channel spectrum")
    plot_spectrum(np.real(shifted), fs=fs_iq, ax=axes[0, 1], title="After tuning")
    plot_waveform(np.real(filtered[:12000]), fs=fs_iq, ax=axes[1, 0], title="Filtered baseband")
    plot_spectrum(audio, fs=fs_iq, ax=axes[1, 1], title="Recovered audio")
    axes[0, 0].set_xlim(0, 120_000)
    axes[0, 1].set_xlim(0, 80_000)
    axes[1, 1].set_xlim(0, 6000)
    for ax in (axes[0, 0], axes[0, 1], axes[1, 1]):
        ax.set_ylim(-100, 5)
    fig.canvas.draw_idle()
    refresh_audio_widget(audio_out, resample_signal(audio, fs_iq, 44_100), rate=44_100)

controls = widgets.interactive(
    update_receiver,
    tune_offset=float_slider(min_value=-80_000, max_value=80_000, step=1000, value=45_000, description="Tune Hz", readout_format=".0f"),
    channel_bw=float_slider(min_value=8_000, max_value=30_000, step=500, value=12_500, description="BW Hz", readout_format=".0f"),
)
display(controls, audio_out)


## Key Takeaway

An SDR receiver is just DSP applied to captured samples. Once you can tune, filter, and demodulate a file, live hardware becomes an input source problem rather than a fundamentally new architecture.